## Príprava dát

Súbor: priprava_dat.ipynb

Program: Hospodárska informatika

Vypracovala: Bc. Terézia Drengubiaková

Bakalárska práca: : Metódy strojového učenia pre včasnú predikciu geomagnetických búrok

Vedúci diplomovej práce: doc. Ing. Peter Butka, PhD.

Konzultanti: Ing. Viera Krešňáková, PhD., RNDr. Šimon Mackovjak, PhD.

### Import knižnice

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

### Načítanie dát z datasetu

In [2]:
df = pd.read_csv('DP_omni.csv')

### Odstránenie nepotrebných atribútov

### Dĺžka datasetu

In [3]:
df['DST'] = df['dst']

In [4]:
len(df)

552568

### Počet záznamov a atribútov

In [5]:
df.shape

(552568, 6)

### Počet chýbajúcich hodnôt

In [6]:
df.isna().sum()

Unnamed: 0    0
time1         0
bz_gsm        0
v             0
dst           0
DST           0
dtype: int64

### Ukážka datesetu

In [7]:
df["bz_gsm"] = df["bz_gsm"].replace(999.9, np.nan)
df["v"] = df["v"].replace(9999.0, np.nan)

In [8]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# --- uisti sa, že time1 je stĺpec (ak je index, vráť ho späť) ---
if "time1" not in df.columns and df.index.name == "time1":
    df = df.reset_index()

# --- (ak sa čas volá "Time", premapuj) ---
if "time1" not in df.columns and "Time" in df.columns:
    df = df.rename(columns={"Time": "time1"})

# 1) fill hodnoty -> NaN
df["bz_gsm"] = df["bz_gsm"].replace(999.9, np.nan)
df["v"]      = df["v"].replace(9999.0, np.nan)

# 2) čas -> datetime + numerická reprezentácia
df["time1"] = pd.to_datetime(df["time1"], utc=True, errors="coerce")
df["t_num"] = df["time1"].astype("int64") / 1e9
df["t_num"] = (df["t_num"] - df["t_num"].mean()) / df["t_num"].std()

# 3) regresná imputácia bz_gsm z (t_num, dst, v)
mask_train_bz = df["bz_gsm"].notna() & df["DST"].notna() & df["v"].notna() & df["t_num"].notna()
mask_pred_bz  = df["bz_gsm"].isna()  & df["DST"].notna() & df["v"].notna() & df["t_num"].notna()

if mask_pred_bz.any() and mask_train_bz.any():
    X_train = df.loc[mask_train_bz, ["t_num", "dst", "v"]].values
    y_train = df.loc[mask_train_bz, "bz_gsm"].values
    model_bz = LinearRegression().fit(X_train, y_train)
    df.loc[mask_pred_bz, "bz_gsm"] = model_bz.predict(df.loc[mask_pred_bz, ["t_num", "dst", "v"]].values)

# 4) regresná imputácia v z (t_num, dst, bz_gsm)
mask_train_v = df["v"].notna() & df["DST"].notna() & df["bz_gsm"].notna() & df["t_num"].notna()
mask_pred_v  = df["v"].isna()  & df["DST"].notna() & df["bz_gsm"].notna() & df["t_num"].notna()

if mask_pred_v.any() and mask_train_v.any():
    X_train = df.loc[mask_train_v, ["t_num", "DST", "bz_gsm"]].values
    y_train = df.loc[mask_train_v, "v"].values
    model_v = LinearRegression().fit(X_train, y_train)
    df.loc[mask_pred_v, "v"] = model_v.predict(df.loc[mask_pred_v, ["t_num", "DST", "bz_gsm"]].values)

# 5) fallback: časová interpolácia (musí byť DatetimeIndex)
df = df.sort_values("time1").set_index("time1")
df["bz_gsm"] = df["bz_gsm"].interpolate(method="time", limit_direction="both")
df["v"]      = df["v"].interpolate(method="time", limit_direction="both")
df = df.reset_index()

# uprac pomocný stĺpec
df = df.drop(columns=["t_num"])


### Atribúty datasetu

In [9]:
df.columns

Index(['time1', 'Unnamed: 0', 'bz_gsm', 'v', 'dst', 'DST'], dtype='object')

### Vytvorenie nových atribútov na predikciu DST

In [10]:
# posúvame vždy o 1 hodnotu menej, nakoľko budeme používať TimeSeries Generatro, ktorý urobí ešte posun o 1
df["DST+1"] = df.DST.shift(periods = 0)
df["DST+2"] = df.DST.shift(periods = -1)
df["DST+3"] = df.DST.shift(periods = -2)
df["DST+4"] = df.DST.shift(periods = -3)
df["DST+5"] = df.DST.shift(periods = -4)
df["DST+6"] = df.DST.shift(periods = -5)

### Kontrola atribútov

In [11]:
df.columns

Index(['time1', 'Unnamed: 0', 'bz_gsm', 'v', 'dst', 'DST', 'DST+1', 'DST+2',
       'DST+3', 'DST+4', 'DST+5', 'DST+6'],
      dtype='object')

### Uloženie upraveného datasetu

In [12]:
df.to_csv("allDST_omni.csv")